# Day 4: same network, but with PyTorch doing the hard math now

days 1-3 were all hand-written numpy, forward pass and backward pass and gradient descent all derived by hand. today's about switching to an actual framework (PyTorch) and seeing how much of that manual work gets replaced by a couple lines. same dataset as day 3 (the news headlines), same basic architecture too, so it's a fair comparison.

nothing here should feel like a mystery since we already know what's happening under the hood from building it ourselves first.

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
torch.manual_seed(42) ##adding for reproducibiltiy and avoiding randomization from pytorch seed generator

df = pd.read_csv("data/news_dataset_cleaned.csv")

X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(
    df["Title"], df["Category"], test_size=0.2, random_state=42
)

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

label_encoder = LabelEncoder()
label_encoder.fit(df["Category"])
y_train = label_encoder.transform(y_train_labels)
y_test = label_encoder.transform(y_test_labels)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (88, 649)
Test shape: (23, 649)


learned the hard way that `torch.manual_seed(42)` is not optional if you want reproducible results -- pytorch has its OWN random number generator, separate from numpy's. forgot this at first and got a different accuracy every single run (52% one time, 43% the next, same exact code) since the weight init, the dataloader shuffling, and dropout are all random unless you pin the seed.

labels here are just plain integers (0-5), not one-hot like day 3, because pytorch's loss function wants it that way (more on that later).

In [2]:
##convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
print("X_train_tensor shape:", X_train_tensor.shape)
print("X_train_tensor dtype:", X_train_tensor.dtype)

X_train_tensor shape: torch.Size([88, 649])
X_train_tensor dtype: torch.float32


a tensor is basically pytorch's version of a numpy array, except pytorch can track everything done to it so it can figure out gradients automatically later. float32 for the actual features, long (64 bit int) specifically for the labels since that's what the loss function requires.

In [3]:
##DataLoader and Dataset
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset): ##wraps our tensors so PyTorch knows how to loop through them
    def __init__(self, X, y): ##just stores the features and labels when the object is created
        self.X = X
        self.y = y

    def __len__(self): ##how many total samples there are
        return len(self.X)

    def __getitem__(self, idx): ##given an index, hand back that one sample's features + label
        return self.X[idx], self.y[idx]

train_dataset = NewsDataset(X_train_tensor, y_train_tensor)
test_dataset = NewsDataset(X_test_tensor, y_test_tensor)

##DataLoader groups the dataset into batches of 16 and reshuffles the training set every epoch
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) ##no need to shuffle test data, order doesn't matter for evaluation

##just peeking at one batch to make sure the shapes look right (16 samples, 649 features / 16 labels)
for batch_X, batch_y in train_loader:
    print("Batch X shape:", batch_X.shape)
    print("Batch y shape:", batch_y.shape)
    break

Batch X shape: torch.Size([16, 649])
Batch y shape: torch.Size([16])


`Dataset` is just a standard way to package up features + labels so pytorch knows how to grab one sample at a time. `DataLoader` takes that and handles batching + shuffling automatically. batch_size=16 means training actually happens in mini batches now, not one giant full-batch pass like every single day up to this point. also messed up the class name at first (typo'd it as `NewsDataser` and also wrote `def_init_` instead of `def __init__`, missing the double underscores and a space) which broke everything until fixed.

In [4]:
##Model (this replaces the hand written forward() function we wrote in the last notebook
import torch.nn as nn

class NewsClassifier(nn.Module): ##nn.Module automatically tracks every layer below as a trainable parameter, no manual list needed like day 3
    def __init__(self, input_size, num_classes):
        super().__init__() ##required boilerplate, sets up nn.Module's internal tracking

        ##same architecture as day 3: 649 -> 32 (batchnorm+relu+dropout) -> 16 (relu) -> 6 output classes
        self.layer1 = nn.Linear(input_size, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.layer2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(16, num_classes) ##no softmax here on purpose, CrossEntropyLoss does that internally

    def forward(self, x): ##called automatically whenever we do model(x), defines how data flows through the layers above
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.layer3(x)
        return x

num_classes = len(label_encoder.classes_)
model = NewsClassifier(input_size=X_train.shape[1], num_classes=num_classes)
print(model)

NewsClassifier(
  (layer1): Linear(in_features=649, out_features=32, bias=True)
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu1): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (layer2): Linear(in_features=32, out_features=16, bias=True)
  (relu2): ReLU()
  (layer3): Linear(in_features=16, out_features=6, bias=True)
)


same exact architecture as day 3's from-scratch version, just using real layers (`nn.Linear`, `nn.BatchNorm1d`, etc) instead of hand-built weight matrices. `nn.Module` automatically keeps track of every layer as a trainable parameter, so no more manually listing out W1, b1, gamma1, beta1, W2... like day 3 needed.

important: no softmax at the end. that's on purpose, not forgotten -- the loss function below does that internally, adding it here too would apply it twice and mess up the math.

In [5]:
##Loss function and optimizer
criterion = nn.CrossEntropyLoss() ##combines softmax + cross entropy in one numerically stable step, expects raw logits + integer labels
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) ##model.parameters() hands over every trainable weight/bias/batchnorm param automatically

`nn.CrossEntropyLoss()` is literally the same math we derived by hand on day 3, just built in. `model.parameters()` is the real payoff of using `nn.Module` -- one call hands the optimizer every single trainable number in the whole network instead of listing 8 separate parameter groups out by hand.

In [6]:
##training loop
epochs = 100
model.train() ##puts dropout/batchnorm into "training mode" behavior

for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader: ##loops through mini-batches, not the whole dataset at once like days 1-3
        optimizer.zero_grad() ##gradients accumulate by default in pytorch, so clear old ones before each new batch
        outputs = model(batch_X) ##forward pass, calls our forward() method automatically
        loss = criterion(outputs, batch_y)
        loss.backward() ##autograd magic -- computes every gradient in the whole network automatically
        optimizer.step() ##applies the adam update to every parameter using those gradients
        total_loss += loss.item() ##.item() pulls the plain python number out of the loss tensor

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: loss={total_loss / len(train_loader):.4f}") ##average loss across all the mini-batches this epoch

Epoch 10: loss=0.2375
Epoch 20: loss=0.0590
Epoch 30: loss=0.0358
Epoch 40: loss=0.0629
Epoch 50: loss=0.0135
Epoch 60: loss=0.1393
Epoch 70: loss=0.1183
Epoch 80: loss=0.0252
Epoch 90: loss=0.0057
Epoch 100: loss=0.0071


this is the part that really shows the difference. `loss.backward()` is one line replacing day 3's ENTIRE hand-derived backward pass, batchnorm math and all. `optimizer.step()` replaces every one of the 8 manual adam_update() calls from before.

`optimizer.zero_grad()` is a genuine pytorch gotcha -- gradients pile up on top of each other by default instead of resetting each step, so you have to manually clear them every batch or things break silently.

also accidentally pasted this whole training loop in twice at first, so it trained for 200 epochs instead of 100 without meaning to. and the accuracy calc down below had a typo (`batch_y.sum().item` instead of `(predictions == batch_y).sum().item()`) that made test accuracy come out as a flat 0.0 the first time. fixed both.

In [7]:
##evaluation loop
model.eval() ##turns dropout off, tells batchnorm to use its saved running stats instead of this batch's

correct = 0
total = 0

with torch.no_grad(): ##don't bother tracking gradients, we're not training here
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        predictions = torch.argmax(outputs, dim=1) ##picks the highest scoring class per sample, same idea as day 3's np.argmax
        correct += (predictions == batch_y).sum().item() ##compares predictions to the true labels element-wise, counts how many matched, pulls out the plain number
        total += batch_y.size(0) ##adds however many samples were in this batch to the running total

test_accuracy = correct / total ##correct guesses divided by total guesses
print("Test accuracy:", test_accuracy)

Test accuracy: 0.5217391304347826


In [8]:
##saving model weights
torch.save(model.state_dict(), "news_classifier_weights.pth")
print("Saved model weights to news_classifier_weights.pth")

Saved model weights to news_classifier_weights.pth


`model.state_dict()` grabs every learned number in the model by name, `torch.save()` writes it to a file. kind of like `joblib.dump` from week 2 day 6 but for a neural net specifically -- only saves the numbers though, not the architecture, so you'd need the `NewsClassifier` class defined again to actually load it back in later.

## takeaway

test accuracy landed at 52.17%, better than day 3's from-scratch version (39%) but still worse than every classical model from week 2 (decision tree and gradient boosting both got 70%). training loss went basically to 0 again too (0.0071 by the end), so still overfitting hard, same story as day 3.

not totally sure why this version did better than day 3's, my best guess is it's because this one trained on all 88 rows directly with no validation split carved out (day 3 held back 18 rows for early stopping, leaving only 70 to actually train on), plus it used mini-batches instead of one big full-batch update every epoch, so the model got a lot more individual gradient updates per epoch this time around. not something i proved for sure, just a guess based on what's different between the two setups.

either way, doesn't change the bigger point from day 3: neural nets still aren't beating simple tree-based models on this particular dataset, there's just not enough data (only 88 rows total) for a model with this many parameters to really shine. framework or no framework, hand-written or not, that part hasn't changed.